In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_csv(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\CCR_DATA_2023_25\Raw_data_1Day_2024_site_124_R_K_Puram_Delhi_DPCC_1Day.csv")

In [3]:
df

,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),NO (µg/m³),NO2 (µg/m³),NOx (ppb),NH3 (µg/m³),SO2 (µg/m³),CO (mg/m³),Ozone (µg/m³),...,MP-Xylene (µg/m³),AT (°C),RH (%),WS (m/s),WD (deg),RF (mm),TOT-RF (mm),SR (W/mt2),BP (mmHg),VWS (m/s)
0,2024-01-01,209.21,326.41,7.74,55.65,35.90,23.63,23.51,1.12,28.90,...,NaN,11.13,69.91,0.51,189.14,NaN,0.0,42.63,991.85,1.08
1,2024-01-02,218.01,337.74,18.54,51.33,42.42,20.67,16.45,0.98,28.47,...,NaN,10.07,67.38,0.48,184.31,NaN,0.0,52.61,991.89,1.74
2,2024-01-03,219.69,338.84,24.00,51.97,46.95,25.57,19.37,1.47,19.42,...,NaN,9.29,78.00,0.46,188.67,NaN,0.0,35.56,993.01,1.90
3,2024-01-04,246.96,395.88,15.68,25.13,25.94,22.74,18.15,1.49,11.47,...,NaN,9.07,79.86,0.54,183.90,NaN,0.0,17.79,992.29,1.88
4,2024-01-05,200.01,328.11,19.12,38.23,35.92,33.27,20.56,1.47,12.25,...,NaN,10.76,81.76,0.37,182.59,NaN,0.0,14.99,992.58,1.55
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
361,2024-12-27,158.00,215.91,44.27,44.48,59.55,34.85,17.45,1.39,9.71,...,NaN,16.19,90.19,0.57,181.60,NaN,0.0,5.78,773.08,-0.02
362,2024-12-28,112.64,132.94,29.81,42.83,46.34,31.35,13.23,0.74,10.31,...,NaN,16.45,92.30,0.36,185.76,NaN,0.0,12.10,NaN,-0.01
363,2024-12-29,108.04,140.09,4.27,34.18,21.27,24.11,2.01,0.75,12.53,...,NaN,15.72,87.66,0.93,195.57,NaN,0.0,44.71,NaN,-0.09
364,2024-12-30,115.46,151.53,9.31,36.70,26.01,20.57,19.89,0.61,11.93,...,NaN,13.37,84.33,0.94,195.45,NaN,0.0,32.15,NaN,-0.08


In [4]:
# ---------- 2. Remove duplicate rows and columns ----------
df = df.drop_duplicates().reset_index(drop=True)
df = df.loc[:, ~df.T.duplicated()]
print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (366, 21)


In [5]:
# ---------- 3. Handle missing values (drop >70% NaN, impute median otherwise) ----------
nan_thresh = 0.7

# Drop columns with >70% missing
cols_to_drop = df.columns[df.isnull().mean() > nan_thresh]
df = df.drop(columns=cols_to_drop)
print(f"Dropped columns (>{int(nan_thresh*100)}% NaN): {cols_to_drop.tolist()}")

# Drop rows with >70% missing
rows_to_drop = df.index[df.isnull().mean(axis=1) > nan_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
print(f"Dropped rows (>{int(nan_thresh*100)}% NaN):", len(rows_to_drop))

# Impute remaining missing values
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns

for col in num_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

for col in cat_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode(dropna=True)
        if not mode_val.empty:
            df[col] = df[col].fillna(mode_val[0])

print("Missing values after imputation:\n", df.isnull().sum())

Dropped columns (>70% NaN): ['Xylene (µg/m³)']
Dropped rows (>70% NaN): 0
Missing values after imputation:
 Timestamp          0
PM2.5 (µg/m³)      0
PM10 (µg/m³)       0
NO (µg/m³)         0
NO2 (µg/m³)        0
NOx (ppb)          0
NH3 (µg/m³)        0
SO2 (µg/m³)        0
CO (mg/m³)         0
Ozone (µg/m³)      0
Benzene (µg/m³)    0
Toluene (µg/m³)    0
AT (°C)            0
RH (%)             0
WS (m/s)           0
WD (deg)           0
TOT-RF (mm)        0
SR (W/mt2)         0
BP (mmHg)          0
VWS (m/s)          0
dtype: int64


In [6]:
# ---------- 4. Handle outliers using IQR (with 70% rule) ----------

def get_outlier_mask(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

# Handle columns: drop if >70% outliers, else replace with median
outlier_thresh = 0.7
cols_to_drop = []
for col in num_cols:
    outlier_mask = get_outlier_mask(df[col])
    outlier_fraction = outlier_mask.mean()
    if outlier_fraction > outlier_thresh:
        cols_to_drop.append(col)
    else:
        median_val = df[col].median()
        df.loc[outlier_mask, col] = median_val
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped numeric columns (>{int(outlier_thresh*100)}% outliers): {cols_to_drop}")
    
# Handle rows: drop if >70% numeric columns are outliers in a given row
outlier_matrix = df[num_cols].apply(get_outlier_mask)
row_outlier_fraction = outlier_matrix.mean(axis=1)
rows_to_drop = df.index[row_outlier_fraction > outlier_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
if len(rows_to_drop) > 0:
    print(f"Dropped rows (>{int(outlier_thresh*100)}% outliers): {len(rows_to_drop)}")


In [7]:
# ---------- 6. Final check ----------
print("Final shape:", df.shape)
print(df.head())

Final shape: (366, 20)
    Timestamp  PM2.5 (µg/m³)  PM10 (µg/m³)  NO (µg/m³)  NO2 (µg/m³)  \
0  2024-01-01         209.21        326.41        7.74        55.65   
1  2024-01-02         218.01        337.74       18.54        51.33   
2  2024-01-03         219.69        338.84       24.00        51.97   
3  2024-01-04         246.96        395.88       15.68        25.13   
4  2024-01-05         200.01        328.11       19.12        38.23   

   NOx (ppb)  NH3 (µg/m³)  SO2 (µg/m³)  CO (mg/m³)  Ozone (µg/m³)  \
0      35.90        23.63        23.51        1.12          28.90   
1      42.42        20.67        16.45        0.98          28.47   
2      46.95        25.57        19.37        1.47          19.42   
3      25.94        22.74        18.15        1.49          11.47   
4      35.92        33.27        20.56        1.47          12.25   

   Benzene (µg/m³)  Toluene (µg/m³)  AT (°C)  RH (%)  WS (m/s)  WD (deg)  \
0            0.175             0.65    11.13   69.91      0

In [8]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
numerics = df.select_dtypes(include=[np.number]).columns
df[numerics] = scaler.fit_transform(df[numerics])

In [9]:
df

,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),NO (µg/m³),NO2 (µg/m³),NOx (ppb),NH3 (µg/m³),SO2 (µg/m³),CO (mg/m³),Ozone (µg/m³),Benzene (µg/m³),Toluene (µg/m³),AT (°C),RH (%),WS (m/s),WD (deg),TOT-RF (mm),SR (W/mt2),BP (mmHg),VWS (m/s)
0,2024-01-01,1.569368,1.126731,-0.688004,0.990502,-0.201632,0.219469,0.844744,-0.284671,-0.849990,0.100682,-0.076256,-1.832946,0.333523,0.186917,0.553356,0.0,-0.535611,-0.061975,2.104855
1,2024-01-02,1.701035,1.233627,-0.217236,0.763882,0.007142,-0.076095,-0.206096,-0.469944,-0.871445,0.100682,-0.076256,-1.971761,0.160654,0.024290,0.048371,0.0,-0.272946,-0.061975,0.194519
2,2024-01-03,1.726171,1.244006,0.020763,0.797456,0.152195,0.413183,0.228529,0.178512,-1.323002,0.100682,-0.076256,-2.073908,0.886295,-0.084128,0.504216,0.0,-0.721687,-0.061975,0.194519
3,2024-01-04,2.134188,1.782170,-0.341903,-0.610522,-0.520557,0.130600,0.046939,0.204979,-1.719673,0.100682,-0.076256,-2.102719,1.013385,0.349544,0.005505,0.0,-1.189378,-0.061975,0.194519
4,2024-01-05,1.431717,1.142770,-0.191955,0.076680,-0.200992,1.182050,0.405654,0.178512,-1.680754,0.100682,-0.076256,-1.881400,1.143207,-0.572008,-0.131458,0.0,-1.263072,-0.061975,0.194519
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
361,2024-12-27,0.803158,0.084179,0.904324,0.404544,0.555654,1.339818,-0.057252,0.072641,-1.807489,-1.362661,-1.226089,-1.170300,1.719211,0.512171,-0.234964,0.0,-1.505471,-0.061975,0.228633
362,2024-12-28,0.124477,-0.698631,0.274019,0.317988,0.132662,0.990333,-0.685375,-0.787556,-1.777552,-1.451348,-1.309713,-1.136251,1.863382,-0.626217,0.199971,0.0,-1.339134,-0.061975,0.245689
363,2024-12-29,0.055651,-0.631171,-0.839260,-0.135776,-0.670093,0.267398,-2.355408,-0.774322,-1.666783,-1.451348,-1.246995,-1.231850,1.546341,2.463695,1.225623,0.0,-0.480867,-0.061975,0.109237
364,2024-12-30,0.166670,-0.523237,-0.619569,-0.003581,-0.518316,-0.086081,0.305928,-0.959595,-1.696721,-0.919223,-0.745249,-1.539601,1.318810,2.517904,1.213077,0.0,-0.811435,-0.061975,0.126293


In [10]:
df.to_excel('RKPuram2024.xlsx', index=False)